# Notebook 07 — Seaborn Pairplots of PCA Components by Regime

Three pairplots are generated — one for each label source:
1. **Unsupervised (balanced_cluster)** — k-means cluster assignment
2. **Grok market_code** — LLM-assisted external labels (if available)
3. **Supervised predicted** — current-regime RF classifier predictions

**Note:** `sns.pairplot` is slow (30–90 seconds for 5 components × 300 rows).
This notebook is kept separate from the main pipeline so it never blocks a headless run.

**Prerequisites:** Run steps 1–5 (or at least steps 1–3 for pairplots 1 and 2).
```
python run_pipeline.py --steps 1,2,3,4,5
```

In [ ]:
%matplotlib inline
import sys
import pickle
from pathlib import Path

repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from market_regime import DATA_DIR, OUTPUT_DIR
from market_regime.config import load
from market_regime import plotting

cfg = load()
REGIMES_DIR = DATA_DIR / 'regimes'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = OUTPUT_DIR / 'models'
print('Data dir:', DATA_DIR)

In [ ]:
# Load PCA components and cluster labels
pca_df = pd.read_parquet(REGIMES_DIR / 'pca_components.parquet')
labels_df = pd.read_parquet(REGIMES_DIR / 'cluster_labels.parquet')
labels = labels_df['balanced_cluster']

print(f'PCA components: {pca_df.shape}')
print(f'Labels: {labels.value_counts().to_dict()}')

# Try loading features for supervised prediction
features = None
try:
    features = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
    print(f'Features: {features.shape}')
except FileNotFoundError:
    print('features.parquet not found — supervised pairplot will be skipped')

In [ ]:
regime_names: dict[int, str] = {}
for name_path in [
    DATA_DIR.parent / 'config' / 'regime_labels.yaml',
    REGIMES_DIR / 'regime_names_suggested.yaml',
]:
    if name_path.exists():
        with open(name_path) as f:
            raw = yaml.safe_load(f) or {}
        names = {int(k): v for k, v in raw.items() if not str(k).startswith('#')}
        if names:
            regime_names = names
            break

if not regime_names:
    unique = sorted(labels.dropna().astype(int).unique())
    regime_names = {i: f'Regime {i}' for i in unique}

print('Regime names:', regime_names)

In [ ]:
N_COMPONENTS = 5
pc_cols = [c for c in pca_df.columns if c.startswith('PC')][:N_COMPONENTS]


def make_pairplot(label_series: pd.Series, title: str, filename: str,
                  name_map: dict | None = None) -> None:
    """Draw and save a seaborn pairplot of PCA components coloured by label_series."""
    valid = label_series.dropna()
    df_plot = pca_df.loc[pca_df.index.intersection(valid.index), pc_cols].copy()
    lbl = valid.reindex(df_plot.index)

    if name_map is None:
        unique_ids = sorted(lbl.astype(int).unique())
        name_map = {i: f'R{i}' for i in unique_ids}

    df_plot['Label'] = lbl.map(
        lambda x: name_map.get(int(x), f'R{int(x)}') if pd.notna(x) else '?'
    )

    unique_ids = sorted(lbl.astype(int).unique())
    palette = {
        name_map.get(i, f'R{i}'): plotting.CUSTOM_COLORS[i % len(plotting.CUSTOM_COLORS)]
        for i in unique_ids
    }

    print(f'Drawing: {title} ({len(df_plot)} quarters, {len(pc_cols)} components) ...')
    g = sns.pairplot(
        df_plot,
        hue='Label',
        palette=palette,
        diag_kind='kde',
        plot_kws={'alpha': 0.55, 's': 18, 'edgecolors': 'none'},
        diag_kws={'fill': True, 'alpha': 0.45},
    )
    g.figure.suptitle(title, y=1.02, fontsize=13)

    out_path = OUTPUT_DIR / 'plots' / filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.figure.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'  Saved to {out_path}')
    plt.show()

In [ ]:
# ── Pairplot 1: Unsupervised balanced_cluster labels ──────────────────────────
make_pairplot(
    label_series=labels,
    title='PCA Pairplot — Unsupervised Regimes (balanced_cluster)',
    filename='07_pca_pairplot_unsupervised.png',
    name_map=regime_names,
)

In [ ]:
# ── Pairplot 2: Grok market_code labels (LLM-assisted seed labels) ────────────
if 'market_code' in labels_df.columns:
    mc = labels_df['market_code'].dropna()
    unique_mc = sorted(mc.astype(int).unique())
    mc_names = {i: f'Grok-{i}' for i in unique_mc}
    make_pairplot(
        label_series=mc,
        title='PCA Pairplot — Grok market_code (LLM-assisted labels)',
        filename='07_pca_pairplot_market_code.png',
        name_map=mc_names,
    )
else:
    print('No market_code column in cluster_labels — skipping grok overlay.')
    print("Run with --market-code grok to add grok labels: python run_pipeline.py --market-code grok")

In [ ]:
# ── Pairplot 3: Supervised predicted regime labels ─────────────────────────────
current_model = None
for model_name in ['current_regime.pkl', 'current_regime_classifier.pkl']:
    model_path = MODELS_DIR / model_name
    if model_path.exists():
        with open(model_path, 'rb') as f:
            current_model = pickle.load(f)
        print(f'Loaded model: {model_name} ({type(current_model).__name__})')
        break

if current_model is not None and features is not None:
    X = features.dropna(axis=1, how='any')
    y_pred = pd.Series(
        current_model.predict(X),
        index=X.index,
        name='predicted_regime',
    )
    print(f'Predicted labels: {y_pred.value_counts().to_dict()}')
    make_pairplot(
        label_series=y_pred,
        title='PCA Pairplot — Supervised Predicted Regimes (RF Classifier)',
        filename='07_pca_pairplot_predicted.png',
        name_map=regime_names,
    )
else:
    print('Supervised pairplot skipped: model or features not available.')
    print('  → Run: python run_pipeline.py --steps 5')

In [ ]:
# ── Summary: regime assignment agreement across label sources ─────────────────
print('=== Regime Assignment Summary ===')
print()
print('Unsupervised (balanced_cluster):')
print(labels.value_counts().sort_index().to_string())

if 'market_code' in labels_df.columns:
    print()
    print('Grok market_code:')
    print(labels_df['market_code'].value_counts().sort_index().to_string())

if current_model is not None and features is not None:
    print()
    print('Supervised (RF predicted):')
    print(y_pred.value_counts().sort_index().to_string())

    # Agreement between unsupervised and supervised
    common_idx = labels.index.intersection(y_pred.index)
    agree = (labels.loc[common_idx].astype(int) == y_pred.loc[common_idx].astype(int)).mean()
    print()
    print(f'Unsupervised vs Supervised agreement: {agree:.1%} ({len(common_idx)} quarters)')